# Milestone 2: how much do wrong labels hold the model back?

Our labels come from Google's cards: two headlines in the same card = same story, in different cards = different stories. The error analysis in `milestone2_experiments.ipynb` showed that Google's cards are sometimes wrong (one story split into two cards) or loose (a digest or opinion piece grouped with a news story). This notebook asks two questions:

1. **Training labels:** can we find the wrong labels in train and does removing them give a better model?
2. **Evaluation labels:** what would the val scores be if the models were graded against corrected labels instead of Google's?

Steps:
1. **Cross-checking:** train one model on each half of train; each one scores the other half. Pairs where the model strongly disagrees with the label are flagged.
2. **Hand check of the flags:** a sample of flagged pairs is read to see how often the label is really wrong.
3. **Retrain on cleaned train:** drop the kind of flagged pairs that the hand check confirms are mostly wrong; retrain the bi-encoder and the cross-encoder.
4. **Corrected val (sample):** a random sample of 150 val pairs that at least one model disagrees with is read and given a corrected label, to estimate each model's F1 against corrected labels.

The hand checks are saved in `outputs/milestone2/label_noise/*.csv`. They were done by one annotator (an AI assistant) and spot-checked by the team. If a hand-check file is missing or empty, the notebook saves it for reading and skips the steps that need it.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'pairs').is_dir())
sys.path.insert(0, str(ROOT))
from src.m2_common import *          # data, metrics, training and scoring shared by the Milestone 2 notebooks
from src.embeddings import clean_headline

OUT = ROOT / 'outputs' / 'milestone2' / 'label_noise'
OUT.mkdir(parents=True, exist_ok=True)
pairs = load_pairs()
TRAIN, VAL, TEST = pairs['train'], pairs['val'], pairs['test']

def judged(path, column):
    """A hand-check file, or None if it does not exist yet or is not filled in."""
    if not path.exists():
        return None
    d = pd.read_csv(path, keep_default_na=False)
    return d if (d[column].astype(str).str.strip() != '').all() else None

def cached_run(kind, tag, train_fn):
    """Reuse a model trained by an earlier run of this notebook."""
    path = MODELS / 'exp' / tag
    if path.exists():
        print(f'{tag}: using saved model')
        return {'kind': kind, 'path': path, 'prep': clean_headline}
    return train_fn()

print('device:', DEVICE, {s: len(d) for s, d in pairs.items()})

W1025 20:06:25.575000 17872 torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


device: cuda {'train': 21907, 'val': 2850, 'test': 2457}


## 1. Cross-checking the training labels

Train is split in two halves by page (snapshot), so the two headlines of a pair always land in the same half. A model trained on one half scores the other half. A pair is **flagged** when the model's score is on the wrong side of its val threshold: a "same story" pair scored below it, or a "different story" pair scored above it. `margin` says how far on the wrong side.

In [2]:
FLAGS = OUT / 'train_flags.csv'
if FLAGS.exists():
    flags = pd.read_csv(FLAGS, dtype={'timestamp': str})
else:
    snaps = TRAIN.timestamp.unique()
    half = dict(zip(snaps, np.random.default_rng(SEED).permutation(len(snaps)) % 2))
    fold = TRAIN.timestamp.map(half).values
    parts = []
    for k in [0, 1]:
        run = cached_run('bi', f'fold{k}', lambda: train_biencoder(TRAIN[fold == k], f'fold{k}', VAL))
        t = best_threshold(VAL.label.values, score_model(run, VAL))
        other = TRAIN[fold != k].copy()
        other['score'], other['threshold'] = score_model(run, other), t
        parts.append(other)
    flags = pd.concat(parts)
    flags['margin'] = np.where(flags.label == 1, flags.threshold - flags.score, flags.score - flags.threshold)
    flags = flags.sort_values('margin', ascending=False)
    flags.to_csv(FLAGS, index=False)

flags['flagged'] = flags.margin > 0
print(flags.groupby('label').flagged.agg(['sum', 'mean']).rename(index={0: 'different story', 1: 'same story'},
      columns={'sum': 'flagged pairs', 'mean': 'share flagged'}).round(3))

                 flagged pairs  share flagged
label                                        
different story           1139          0.099
same story                1191          0.114


In [3]:
SAMPLE = OUT / 'train_flags_sample.csv'
if not SAMPLE.exists():
    strong = flags[flags.margin > 0.05]
    sample = pd.concat([strong[strong.label == lab].sample(40, random_state=SEED) for lab in [1, 0]])
    cols = ['label', 'edition', 'score', 'margin', 'headline_a', 'outlet_a', 'headline_b', 'outlet_b', 'timestamp']
    sample[cols].assign(score=lambda d: d.score.round(3), margin=lambda d: d.margin.round(3), label_wrong='', note='').to_csv(SAMPLE, index=False)
    print('saved for hand check:', SAMPLE.relative_to(ROOT))
print(f"pairs with margin > 0.05: same story {((flags.margin > 0.05) & (flags.label == 1)).sum():,}, "
      f"different story {((flags.margin > 0.05) & (flags.label == 0)).sum():,}")

pairs with margin > 0.05: same story 538, different story 407


## 2. Hand check of the flagged pairs

40 flagged "same story" pairs and 40 flagged "different story" pairs (margin above 0.05), picked at random. `label_wrong`: `yes` = Google's label is wrong, `no` = the label is right (the model is wrong), `unclear` = same broad topic, different events.

**Rule:** a kind of flagged pair (same-story or different-story) is dropped from train when at least half of its sample has a wrong label (`yes`). Dropping is safer than flipping the label, since `unclear` pairs are common.

In [4]:
hand = judged(SAMPLE, 'label_wrong')
DROP_SIDES = []
if hand is None:
    print('Hand check not filled in yet:', SAMPLE.relative_to(ROOT))
else:
    hand['label'] = hand.label.map({1: 'same story', 0: 'different story'})
    display(pd.crosstab(hand.label, hand.label_wrong, margins=True))
    share = hand.groupby('label').label_wrong.apply(lambda s: (s == 'yes').mean())
    print((share * 100).round(0).astype(int).astype(str) + '% of flagged pairs have a wrong label')
    DROP_SIDES = [1 if side == 'same story' else 0 for side, s in share.items() if s >= 0.5]
    print('drop flagged pairs with label:', DROP_SIDES or 'none')

label_wrong,no,unclear,yes,All
label,,,,
different story,19,11,10,40
same story,20,10,10,40
All,39,21,20,80


label
different story    25% of flagged pairs have a wrong label
same story         25% of flagged pairs have a wrong label
Name: label_wrong, dtype: object
drop flagged pairs with label: none


## 3. Retraining on the cleaned train set

Same recipes as v1 (bi-encoder) and the cross-encoder in `milestone2_experiments.ipynb`; only the training pairs change.

In [5]:
RUNS = {'pretrained': {'kind': 'bi', 'path': BASE_MODEL, 'prep': clean_headline},
        'v1': {'kind': 'bi', 'path': MODELS / 'minilm_finetuned', 'prep': clean_headline},
        'cross_encoder': {'kind': 'cross', 'path': MODELS / 'exp' / 'cross_encoder', 'prep': clean_headline}}
if hand is not None:
    drop = flags[(flags.margin > 0.05) & flags.label.isin(DROP_SIDES)]
    key = lambda d: d.timestamp + '|' + d.headline_a + '|' + d.headline_b
    cleaned = TRAIN[~key(TRAIN).isin(set(key(drop)))]
    print(f'cleaned train: {len(cleaned):,} pairs ({len(TRAIN) - len(cleaned):,} dropped)')
    if DROP_SIDES:
        RUNS['v1_cleaned'] = cached_run('bi', 'v1_cleaned', lambda: train_biencoder(cleaned, 'v1_cleaned', VAL))
        RUNS['cross_encoder_cleaned'] = cached_run('cross', 'cross_encoder_cleaned',
                                                   lambda: train_crossencoder(cleaned, 'cross_encoder_cleaned', VAL))
print('models compared:', list(RUNS))

cleaned train: 21,907 pairs (0 dropped)
models compared: ['pretrained', 'v1', 'cross_encoder']


## 4. Corrected val labels (sample)

Each model's val threshold is chosen on Google's labels, as before. A val pair is **disputed** when at least one model disagrees with Google's label. A random sample of 150 disputed pairs is read by hand (`val_check_sample.csv`; `same_story`: `yes`, `no`, or `unclear`). Pairs that every model agrees with keep Google's label.

The corrected F1 is then **estimated**: agreed pairs count with Google's label, and each hand-checked disputed pair stands for (disputed pairs / 150) pairs, with `unclear` pairs left out. The 95% interval comes from resampling the 150 checked pairs (bootstrap). Re-checking every disputed pair, and the test set, was left out for time.

In [6]:
def disputed(df, scores, thresholds):
    wrong = np.zeros(len(df), int)
    for name, s in scores.items():
        wrong += ((s >= thresholds[name]).astype(int) != df.label.values)
    return wrong

VAL_SCORES = {name: score_model(run, VAL) for name, run in RUNS.items()}
T_GOOGLE = {name: best_threshold(VAL.label.values, s) for name, s in VAL_SCORES.items()}
n_wrong = disputed(VAL, VAL_SCORES, T_GOOGLE)
DISPUTED = VAL.index[n_wrong > 0]
print(f'val pairs disputed by at least one model: {len(DISPUTED):,} of {len(VAL):,}')
VAL_CHECK = OUT / 'val_check_sample.csv'
if hand is not None and not VAL_CHECK.exists():
    pick = np.sort(np.random.default_rng(SEED).choice(DISPUTED, size=min(150, len(DISPUTED)), replace=False))
    VAL.loc[pick].assign(pair_id=pick, models_disagreeing=n_wrong[pick])[
        ['pair_id', 'label', 'models_disagreeing', 'edition', 'headline_a', 'outlet_a', 'headline_b', 'outlet_b', 'timestamp']
    ].assign(same_story='', note='').to_csv(VAL_CHECK, index=False)
    print('saved for hand check:', VAL_CHECK.relative_to(ROOT))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

val pairs disputed by at least one model: 627 of 2,850


In [7]:
vcheck = judged(VAL_CHECK, 'same_story') if hand is not None else None
if vcheck is None:
    print('val hand check not filled in yet')
else:
    weight = len(DISPUTED) / len(vcheck)                 # disputed pairs each checked pair stands for
    known = vcheck[vcheck.same_story != 'unclear']
    s_idx = VAL.index.get_indexer(known.pair_id)
    s_true = known.same_story.map({'yes': 1, 'no': 0}).values
    agreed = VAL.index.get_indexer(VAL.index.difference(DISPUTED))
    flips = (s_true != VAL.label.values[s_idx])
    print(f"checked {len(vcheck)} disputed pairs: Google's label wrong in {flips.sum()} ({flips.sum() / len(vcheck):.0%}), "
          f"unclear {(vcheck.same_story == 'unclear').sum()} ({(vcheck.same_story == 'unclear').mean():.0%}), right in the rest")
    print(f"-> estimated share of all val labels that are wrong: {flips.sum() / len(vcheck) * len(DISPUTED) / len(VAL):.1%}")

    def corrected_f1(name, rows):
        s, t = VAL_SCORES[name], T_GOOGLE[name]
        pa, ya = (s[agreed] >= t), VAL.label.values[agreed] == 1
        ps, ys = (s[s_idx[rows]] >= t), s_true[rows] == 1
        tp = (pa & ya).sum() + weight * (ps & ys).sum()
        fp = (pa & ~ya).sum() + weight * (ps & ~ys).sum()
        fn = (~pa & ya).sum() + weight * (~ps & ys).sum()
        return 2 * tp / (2 * tp + fp + fn)

    rng = np.random.default_rng(SEED)
    boots = [rng.integers(0, len(known), len(known)) for _ in range(2000)]
    rows = []
    for name, s in VAL_SCORES.items():
        est = corrected_f1(name, np.arange(len(known)))
        ci = np.percentile([corrected_f1(name, b) for b in boots], [2.5, 97.5])
        rows.append({'model': name, 'F1 (Google labels)': evaluate(VAL.label.values, s, T_GOOGLE[name])['f1'],
                     'F1 (corrected, estimate)': est, '95% interval': f'{ci[0]:.3f}-{ci[1]:.3f}'})
    val_table = pd.DataFrame(rows).set_index('model')
    val_table.to_csv(OUT / 'val_corrected_f1.csv')
    display(val_table.round(4))

checked 150 disputed pairs: Google's label wrong in 12 (8%), unclear 45 (30%), right in the rest
-> estimated share of all val labels that are wrong: 1.8%


,F1 (Google labels),"F1 (corrected, estimate)",95% interval
model,,,
pretrained,0.8327,0.8830,0.867-0.899
v1,0.8823,0.9322,0.914-0.948
cross_encoder,0.8861,0.9461,0.930-0.962
